# E9 · Variantes del stacking con *bagging* + LightGBM
**Parte de:** E8b (bagging de 5 semillas, 5 modelos + LightGBM; CV 0,8179, público **0,80168**, el mejor hasta ahora).

**Objetivo.** E8b mostró que agregar un modelo no lineal **y además** promediar semillas sí ayuda. Aquí se prueban extensiones de esa idea, con todos los modelos base promediados en 5 semillas:
- Más modelos base: los de E3 (LR-charc, NB-LR-charc, Cadena), que sin bagging no aportaban.
- Quitar CNB, el modelo base más débil.
- Un metamodelo más regularizado (`C=0,3`).
- Umbrales 0,45 (en todas las etiquetas, o solo en HS y AG).

Los conjuntos y umbrales se definieron de antemano (16 combinaciones) para no sobreajustar la selección.

### Glosario de esta prueba
| Término | Significado |
|---|---|
| **HS / TR / AG** | Las 3 etiquetas: discurso de odio / odio dirigido a una persona concreta / agresividad |
| **F1 medio** | F1 de la clase 1 de cada etiqueta, promediado entre HS, TR y AG (métrica principal) |
| **CV repetida** | Validación cruzada de 5 particiones repetida con 3 semillas (42, 43, 44); se reporta media ± desviación |
| **OOF** | *Out-of-fold*: predicción de cada fila del train hecha por un modelo que no la vio al entrenar |
| **Stacking** | Un metamodelo (regresión logística) aprende a combinar las probabilidades OOF de los modelos base |
| **Seed bagging** | Promediar modelos entrenados con distintas particiones aleatorias |
| **LightGBM** | Árboles de decisión potenciados: el único modelo no lineal del stacking |

El glosario completo está en `RESUMEN.md`.

In [1]:
import os, sys, platform
sys.path.insert(0, os.path.abspath('.'))          # el notebook vive en experimentos/, junto a toxic_base.py
import numpy as np, pandas as pd, sklearn, scipy
import toxic_base as tb
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 30); pd.set_option('display.max_colwidth', 120)
print('Python', platform.python_version(), '| numpy', np.__version__, '| pandas', pd.__version__,
      '| scikit-learn', sklearn.__version__, '| scipy', scipy.__version__)
print('Semillas de la CV repetida:', tb.SEEDS, '| particiones:', tb.N_FOLDS)
d = tb.Datos()
print('Datos en', d.data_dir, '| train', d.train.shape, '| test', d.test.shape)

Python 3.12.3 | numpy 2.5.4 | pandas 3.0.6 | scikit-learn 1.9.1 | scipy 1.18.1
Semillas de la CV repetida: (42, 43, 44) | particiones: 5


Datos en /home/jc/repositories/natural-language-processing-workshops/workshops/Kaggle-tweet-toxic/tweet-toxic-2026-puj | train (1731, 5) | test (1154, 2)


In [2]:
from sklearn.linear_model import LogisticRegression
S5 = (42, 43, 44, 45, 46)
B6 = tb.BASES_ORIGINALES + ['LGBM']
EXTRA = ['LR-charc', 'NB-LR-charc', 'Cadena']
for s in S5:
    for b in B6 + EXTRA:
        tb.oof(d, b, s); tb.pred_test_folds(d, b, s)
meta_c03 = lambda: LogisticRegression(C=0.3, max_iter=3000, class_weight='balanced')
CONJ = {'E8b · 5 + LGBM': (B6, tb.meta_lr), '5 + LGBM + charc': (B6 + ['LR-charc', 'NB-LR-charc'], tb.meta_lr),
        '5 + LGBM + Cadena': (B6 + ['Cadena'], tb.meta_lr), 'Los 9 modelos': (B6 + EXTRA, tb.meta_lr),
        'Sin CNB (LR, PS, SVM, NB-LR, LGBM)': (['LR', 'Powerset', 'SVM', 'NB-LR', 'LGBM'], tb.meta_lr)}
R, POOF = {}, {}
for k, (bs, meta) in CONJ.items():
    POOF[k] = {s: tb.stacking_bagged(d, bs, S5, s, meta=meta, con_test=False)[0] for s in tb.SEEDS}
    R[k] = tb.evaluar(d, POOF[k])
    R[k + ' · umbral 0,45'] = tb.evaluar(d, POOF[k], (0.45,) * 3)
    R[k + ' · umbral (0,45; 0,5; 0,45)'] = tb.evaluar(d, POOF[k], (0.45, 0.5, 0.45))
R['E8b · meta C=0,3'] = tb.evaluar(d, {s: tb.stacking_bagged(d, B6, S5, s, meta=meta_c03, con_test=False)[0] for s in tb.SEEDS})
tb.tabla(R).sort_values('F1 medio', ascending=False)

,F1 HS,F1 TR,F1 AG,F1 medio,±,EMR,F1 medio por semilla
Configuración,,,,,,,
Los 9 modelos,0.8107,0.8408,0.8036,0.8184,0.0002,0.7803,0.818 / 0.818 / 0.819
"E8b · meta C=0,3",0.8102,0.8402,0.8045,0.8183,0.0003,0.7801,0.818 / 0.818 / 0.819
"Sin CNB (LR, PS, SVM, NB-LR, LGBM) · umbral (0,45; 0,5; 0,45)",0.8104,0.8390,0.8055,0.8183,0.0014,0.7776,0.818 / 0.817 / 0.820
5 + LGBM + Cadena,0.8099,0.8389,0.8053,0.8180,0.0005,0.7811,0.818 / 0.819 / 0.818
"Sin CNB (LR, PS, SVM, NB-LR, LGBM) · umbral 0,45",0.8104,0.8382,0.8055,0.8180,0.0013,0.7776,0.818 / 0.816 / 0.820
E8b · 5 + LGBM,0.8071,0.8405,0.8062,0.8179,0.0012,0.7785,0.819 / 0.818 / 0.816
"E8b · 5 + LGBM · umbral (0,45; 0,5; 0,45)",0.8106,0.8390,0.8035,0.8177,0.0009,0.7768,0.816 / 0.818 / 0.818
"Los 9 modelos · umbral (0,45; 0,5; 0,45)",0.8080,0.8404,0.8046,0.8177,0.0007,0.7735,0.817 / 0.818 / 0.819
"E8b · 5 + LGBM · umbral 0,45",0.8106,0.8386,0.8035,0.8176,0.0012,0.7776,0.816 / 0.818 / 0.819


**Lectura.** Las 16 variantes quedan entre 0,8164 y 0,8184, todas a ±0,002 de E8b (0,8179): es una **meseta**. Se envían las 3 mejores por CV que además difieren entre sí (otro conjunto de bases, otro metamodelo, otros umbrales), para tener candidatos distintos a la hora de elegir los finales.

## Submissions

In [3]:
CAND = {'toxic_submission_E9a_todos9.csv': (B6 + EXTRA, tb.meta_lr, (0.5, 0.5, 0.5)),
        'toxic_submission_E9b_meta_c03.csv': (B6, meta_c03, (0.5, 0.5, 0.5)),
        'toxic_submission_E9c_sin_cnb_umbral.csv': (['LR', 'Powerset', 'SVM', 'NB-LR', 'LGBM'], tb.meta_lr, (0.45, 0.5, 0.45))}
_, Pt8b = tb.stacking_bagged(d, B6, S5, 42)
e8b = tb.binarizar(Pt8b); e8b[d.err_excel_te] = 0
for nombre, (bs, meta, u) in CAND.items():
    print('=' * 20, nombre)
    _, Pt = tb.stacking_bagged(d, bs, S5, 42, meta=meta)
    sub = tb.guardar_submission(d, tb.binarizar(Pt, u), nombre)
    print('Filas distintas frente a E8b (0,80168):', int((sub[tb.LABELS].values != e8b).any(1).sum()))

==================== toxic_submission_E9a_todos9.csv


[     7s] Guardada /home/jc/repositories/natural-language-processing-workshops/workshops/Kaggle-tweet-toxic/experimentos/submissions/toxic_submission_E9a_todos9.csv


Prevalencia train : {'HS': np.float64(0.437), 'TR': np.float64(0.3), 'AG': np.float64(0.382)}
Prevalencia pred. : {'HS': np.float64(0.401), 'TR': np.float64(0.3), 'AG': np.float64(0.369)}
Cambios frente a la base 0,79160: 64 filas ({'HS': 49, 'TR': 22, 'AG': 21})
Filas distintas frente a E8b (0,80168): 29
==================== toxic_submission_E9b_meta_c03.csv
[     7s] Guardada /home/jc/repositories/natural-language-processing-workshops/workshops/Kaggle-tweet-toxic/experimentos/submissions/toxic_submission_E9b_meta_c03.csv


Prevalencia train : {'HS': np.float64(0.437), 'TR': np.float64(0.3), 'AG': np.float64(0.382)}
Prevalencia pred. : {'HS': np.float64(0.4), 'TR': np.float64(0.304), 'AG': np.float64(0.368)}
Cambios frente a la base 0,79160: 44 filas ({'HS': 26, 'TR': 15, 'AG': 20})
Filas distintas frente a E8b (0,80168): 34
==================== toxic_submission_E9c_sin_cnb_umbral.csv


[     7s] Guardada /home/jc/repositories/natural-language-processing-workshops/workshops/Kaggle-tweet-toxic/experimentos/submissions/toxic_submission_E9c_sin_cnb_umbral.csv


Prevalencia train : {'HS': np.float64(0.437), 'TR': np.float64(0.3), 'AG': np.float64(0.382)}
Prevalencia pred. : {'HS': np.float64(0.425), 'TR': np.float64(0.309), 'AG': np.float64(0.384)}
Cambios frente a la base 0,79160: 66 filas ({'HS': 50, 'TR': 21, 'AG': 26})
Filas distintas frente a E8b (0,80168): 36


## Conclusión
- El stacking con bagging + LightGBM llegó a una meseta en ~0,818 de CV. Las variantes candidatas (E9a, E9b, E9c) tienen la misma calidad esperada que E8b y difieren solo en decenas de filas.
- La diferencia entre ellas en el público será sobre todo ruido. Para la selección final pesa más la CV que el público.